In [5]:
import time
import numpy as np
import sklearn
from sklearn.neighbors import KDTree, BallTree, radius_neighbors_graph
from sklearn.metrics.pairwise import euclidean_distances
from scipy.sparse import csr_array
from scipy.io import mmwrite, mmread

import cppimport
import cppimport.import_hook
from extensions.brute_force import _BruteForce

class BruteForce(object):

    def __init__(self, points):
        self.n = points.shape[0]
        self.d = points.shape[1]
        self.bf = _BruteForce(points)
    
    def radius_neighbors_graph(self, radius, num_threads=4):
        rowptrs, colids, dists = self.bf._radius_neighbors_graph(radius, num_threads)
        return csr_array((dists, colids, rowptrs), shape=(self.n, self.n))

        

n, d = 10000, 16
points = np.random.uniform(0,1,size=(n,d))

brute_force = BruteForce(points)
tree = KDTree(points)


In [6]:
%%timeit
graph = brute_force.radius_neighbors_graph(0.7, num_threads=8)

59.3 ms ± 401 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
graph1 = brute_force.radius_neighbors_graph(0.7, num_threads=1)
graph2 = brute_force.radius_neighbors_graph(0.7, num_threads=8)

In [8]:
print(np.allclose(graph1.todense(), graph2.todense()))

True


In [9]:
graph1

<10000x10000 sparse array of type '<class 'numpy.float64'>'
	with 17144 stored elements in Compressed Sparse Row format>